<a href="https://colab.research.google.com/github/ParthHarde/breast-cancer-project/blob/main/metabric/01_metabric_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = "/content/drive/MyDrive/breast-cancer-project/metabric"
for sub in ["data/raw", "data/processed", "models", "results"]:
    os.makedirs(f"{BASE}/{sub}", exist_ok=True)
print(os.listdir(BASE))

Mounted at /content/drive
['data', 'models', 'results']


In [2]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"parthharde","key":"af3528307f964a92be35e496d56da9d3"}'}

In [3]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [4]:
!kaggle datasets download -d raghadalharbi/breast-cancer-gene-expression-profiles-metabric -p "{BASE}/data/raw" --unzip
!ls -lh "{BASE}/data/raw"

Dataset URL: https://www.kaggle.com/datasets/raghadalharbi/breast-cancer-gene-expression-profiles-metabric
License(s): DbCL-1.0
100% 2.72M/2.72M [00:00<00:00, 131MB/s]

total 8.1M
-rw------- 1 root root 8.1M Oct  3 06:12 METABRIC_RNA_Mutation.csv


In [5]:
import pandas as pd
df = pd.read_csv(f"{BASE}/data/raw/METABRIC_RNA_Mutation.csv", low_memory=False)
print(df.shape)
df.head()

(1904, 693)


,patient_id,age_at_diagnosis,type_of_breast_surgery,cancer_type,cancer_type_detailed,cellularity,chemotherapy,pam50_+_claudin-low_subtype,cohort,er_status_measured_by_ihc,...,mtap_mut,ppp2cb_mut,smarcd1_mut,nras_mut,ndfip1_mut,hras_mut,prps2_mut,smarcb1_mut,stmn2_mut,siah1_mut
0,0,75.65,MASTECTOMY,Breast Cancer,Breast Invasive Ductal Carcinoma,NaN,0,claudin-low,1.0,Positve,...,0,0,0,0,0,0,0,0,0,0
1,2,43.19,BREAST CONSERVING,Breast Cancer,Breast Invasive Ductal Carcinoma,High,0,LumA,1.0,Positve,...,0,0,0,0,0,0,0,0,0,0
2,5,48.87,MASTECTOMY,Breast Cancer,Breast Invasive Ductal Carcinoma,High,1,LumB,1.0,Positve,...,0,0,0,0,0,0,0,0,0,0
3,6,47.68,MASTECTOMY,Breast Cancer,Breast Mixed Ductal and Lobular Carcinoma,Moderate,1,LumB,1.0,Positve,...,0,0,0,0,0,0,0,0,0,0
4,8,76.97,MASTECTOMY,Breast Cancer,Breast Mixed Ductal and Lobular Carcinoma,High,1,LumB,1.0,Positve,...,0,0,0,0,0,0,0,0,0,0


In [6]:
df.info(max_cols=40)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1904 entries, 0 to 1903
Columns: 693 entries, patient_id to siah1_mut
dtypes: float64(498), int64(5), object(190)
memory usage: 10.1+ MB


In [7]:
print(list(df.columns[:35]))

['patient_id', 'age_at_diagnosis', 'type_of_breast_surgery', 'cancer_type', 'cancer_type_detailed', 'cellularity', 'chemotherapy', 'pam50_+_claudin-low_subtype', 'cohort', 'er_status_measured_by_ihc', 'er_status', 'neoplasm_histologic_grade', 'her2_status_measured_by_snp6', 'her2_status', 'tumor_other_histologic_subtype', 'hormone_therapy', 'inferred_menopausal_state', 'integrative_cluster', 'primary_tumor_laterality', 'lymph_nodes_examined_positive', 'mutation_count', 'nottingham_prognostic_index', 'oncotree_code', 'overall_survival_months', 'overall_survival', 'pr_status', 'radio_therapy', '3-gene_classifier_subtype', 'tumor_size', 'tumor_stage', 'death_from_cancer', 'brca1', 'brca2', 'palb2', 'pten']


In [8]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(1)
print(pd.DataFrame({"missing": missing, "percent": missing_pct}).head(15))

                                missing  percent
tumor_stage                         501     26.3
3-gene_classifier_subtype           204     10.7
primary_tumor_laterality            106      5.6
neoplasm_histologic_grade            72      3.8
cellularity                          54      2.8
mutation_count                       45      2.4
er_status_measured_by_ihc            30      1.6
type_of_breast_surgery               22      1.2
tumor_size                           20      1.1
oncotree_code                        15      0.8
tumor_other_histologic_subtype       15      0.8
cancer_type_detailed                 15      0.8
death_from_cancer                     1      0.1
akr1c1                                0      0.0
akr1c2                                0      0.0


In [9]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate patient IDs:", df["patient_id"].duplicated().sum())

Duplicate rows: 0
Duplicate patient IDs: 0


In [10]:
print(df.dtypes.value_counts())   # how many numeric vs text columns

float64    498
object     190
int64        5
Name: count, dtype: int64


In [11]:
print(df.shape)

(1904, 693)


In [12]:
for col in ["overall_survival", "death_from_cancer", "overall_survival_months"]:
    if col in df.columns:
        print(f"\n--- {col} ---")
        print(df[col].value_counts(dropna=False).head(10))


--- overall_survival ---
overall_survival
0    1103
1     801
Name: count, dtype: int64

--- death_from_cancer ---
death_from_cancer
Living                  801
Died of Disease         622
Died of Other Causes    480
NaN                       1
Name: count, dtype: int64

--- overall_survival_months ---
overall_survival_months
192.200000    4
164.333333    3
48.433333     3
34.700000     3
152.066667    3
98.700000     3
117.666667    3
150.600000    3
43.100000     3
38.033333     3
Name: count, dtype: int64


In [13]:
print(pd.crosstab(df["overall_survival"], df["death_from_cancer"], dropna=False))

death_from_cancer  Died of Disease  Died of Other Causes  Living  NaN
overall_survival                                                     
0                              622                   480       0    1
1                                0                     0     801    0


In [14]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate patient IDs:", df["patient_id"].duplicated().sum())
print(df.dtypes.value_counts())

Duplicate rows: 0
Duplicate patient IDs: 0
float64    498
object     190
int64        5
Name: count, dtype: int64


In [15]:
clinical = df.iloc[:, :31]
summary = pd.DataFrame({
    "dtype": clinical.dtypes,
    "unique_values": clinical.nunique(),
    "missing": clinical.isnull().sum()
})
print(summary)

                                  dtype  unique_values  missing
patient_id                        int64           1904        0
age_at_diagnosis                float64           1572        0
type_of_breast_surgery           object              2       22
cancer_type                      object              2        0
cancer_type_detailed             object              6       15
cellularity                      object              3       54
chemotherapy                      int64              2        0
pam50_+_claudin-low_subtype      object              7        0
cohort                          float64              5        0
er_status_measured_by_ihc        object              2       30
er_status                        object              2        0
neoplasm_histologic_grade       float64              3       72
her2_status_measured_by_snp6     object              4        0
her2_status                      object              2        0
tumor_other_histologic_subtype   object 